# Model Comparison

Compare two models structurally and behaviorally: weight distances,
activation divergence, prediction agreement, attention pattern comparison,
and component importance ranking correlation.

## Why This Matters

Model comparison provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_comparison import (
    weight_distance,
    activation_divergence,
    prediction_agreement,
    attention_pattern_comparison,
    component_importance_comparison,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
def make_model(seed):
    m = HookedTransformer(cfg)
    key = jax.random.PRNGKey(seed)
    leaves, treedef = jax.tree.flatten(m)
    new_leaves = []
    for leaf in leaves:
        if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
            key, subkey = jax.random.split(key)
            new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
        else:
            new_leaves.append(leaf)
    return jax.tree.unflatten(treedef, new_leaves)

model_a = make_model(42)
model_b = make_model(99)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Two models ready")

## Weight Distance

In [ ]:
result = weight_distance(model_a, model_b)
print(f"Total weight distance: {result['total_distance']:.4f}")
print(f"Max distance component: {result['max_distance_component']}")
print(f"\nTop relative distances:")
sorted_rel = sorted(result['relative_distances'].items(), key=lambda x: -x[1])
for comp, d in sorted_rel[:5]:
    print(f"  {comp}: {d:.4f}")

## Activation Divergence

In [ ]:
result = activation_divergence(model_a, model_b, tokens)
print(f"Logit divergence: {result['logit_divergence']:.4f}")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: divergence={result['layer_divergence'][i]:.4f}, "
          f"cosine={result['cosine_similarity'][i]:.4f}")

## Prediction Agreement

In [ ]:
tokens_list = [tokens, jnp.array([1, 2, 3, 4, 5, 6, 7]), jnp.array([10, 20, 30, 40, 50, 60, 70])]
result = prediction_agreement(model_a, model_b, tokens_list, top_k=5)
print(f"Agreement rate: {result['agreement_rate']:.1%}")
print(f"Top-5 overlap: {result['top_k_overlap']:.1%}")
print(f"Mean KL divergence: {result['mean_kl']:.4f}")

## Attention Pattern Comparison

In [ ]:
result = attention_pattern_comparison(model_a, model_b, tokens)
print(f"Mean pattern distance: {result['mean_distance']:.4f}")
l, h = result['most_different_head']
print(f"Most different head: L{l}H{h}")
print(f"\nPattern cosine similarity per head:")
for l in range(min(2, cfg.n_layers)):
    for h in range(cfg.n_heads):
        print(f"  L{l}H{h}: {result['pattern_cosine'][l, h]:.4f}")

## Component Importance Comparison

In [ ]:
result = component_importance_comparison(model_a, model_b, tokens, metric_fn)
print(f"Rank correlation: {result['rank_correlation']:.4f}")
print(f"\nBiggest rank changes:")
for comp, ra, rb in result['biggest_rank_changes'][:5]:
    print(f"  {comp}: rank {ra} -> {rb} (delta={abs(ra-rb)})")